# 03 — Preprocess

Transforms simulated multi-source data into the canonical **zone-day** feature table
(`docs/decisions/spatial-analysis-unit.md`), matching `docs/schemas.md` field conventions
where applicable. Three things demonstrated: zone delineation via k-means, a gap-filling
placeholder for cloudy Sentinel-2 scenes, and strict separation of the two weather forcing
lines (operational vs archive) per `docs/decisions/weather-forcing-split.md`.


In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

rng = np.random.default_rng(11)
WINDOW_DAYS = 30
dates = pd.date_range("2025-06-01", periods=WINDOW_DAYS, freq="D", tz="UTC")

def seasonal_signal(n, base, amplitude, noise_std, period=365, phase=150, rng=rng):
    t = np.arange(n)
    signal = base + amplitude * np.sin(2 * np.pi * (t + phase) / period)
    return signal + rng.normal(0, noise_std, size=n)


## Step (a): Zone delineation

Per `docs/decisions/spatial-analysis-unit.md`: k-means over multi-year NDVI plus SoilGrids
gives 2-4 management zones per parcel — not a full 10 m raster, not a single parcel average.
Simulated here over a placeholder set of sub-parcel points in the Aegean pilot AOI.


In [2]:
N_POINTS = 60  # placeholder sub-parcel sample points within one parcel

points_df = pd.DataFrame({
    "point_id": range(N_POINTS),
    "mean_ndvi": rng.normal(0.55, 0.12, N_POINTS).clip(0, 1),
    "clay_pct": rng.normal(28, 8, N_POINTS).clip(0, 100),
    "sand_pct": rng.normal(40, 10, N_POINTS).clip(0, 100),
})

N_ZONES = 3  # within the 2-4 range specified by spatial-analysis-unit.md
features = points_df[["mean_ndvi", "clay_pct", "sand_pct"]].to_numpy()
features_scaled = (features - features.mean(axis=0)) / features.std(axis=0)

kmeans = KMeans(n_clusters=N_ZONES, random_state=0, n_init=10)
points_df["zone_id"] = ["zone-" + str(z) for z in kmeans.fit_predict(features_scaled)]

points_df.groupby("zone_id")[["mean_ndvi", "clay_pct", "sand_pct"]].mean()


,mean_ndvi,clay_pct,sand_pct
zone_id,,,
zone-0,0.408800,24.666656,35.786231
zone-1,0.581029,25.970594,47.022053
zone-2,0.616760,37.519784,36.293169


**Note:** zone definitions refresh annually and stay fixed within a season
(`spatial-analysis-unit.md` consequence) — this k-means run stands in for that annual refresh
step, not something re-run per timestep.


## Step (b): Gap-filling simulated cloudy Sentinel-2 scenes

Placeholder strategy: simple temporal linear interpolation across cloud-masked NDVI gaps.
**This is a placeholder, not a production gap-fill method** — a real implementation would
likely blend in Sentinel-1-informed or climatological priors; that is out of scope here.


In [3]:
ndvi_true = seasonal_signal(WINDOW_DAYS, base=0.55, amplitude=0.15, noise_std=0.02)
cloud_mask = rng.random(WINDOW_DAYS) < 0.35
ndvi_observed = pd.Series(np.where(cloud_mask, np.nan, ndvi_true), index=dates)

# PLACEHOLDER gap-fill: linear interpolation, then edge-fill for leading/trailing NaNs
ndvi_filled = ndvi_observed.interpolate(method="linear").ffill().bfill()

gapfill_df = pd.DataFrame({
    "date": dates, "ndvi_observed": ndvi_observed.values,
    "ndvi_filled": ndvi_filled.values, "was_gap": cloud_mask,
})
gapfill_df.head(10)


,date,ndvi_observed,ndvi_filled,was_gap
0,2025-06-01 00:00:00+00:00,NaN,0.639035,True
1,2025-06-02 00:00:00+00:00,0.639035,0.639035,False
2,2025-06-03 00:00:00+00:00,0.598613,0.598613,False
3,2025-06-04 00:00:00+00:00,NaN,0.602017,True
4,2025-06-05 00:00:00+00:00,0.605420,0.605420,False
5,2025-06-06 00:00:00+00:00,0.603700,0.603700,False
6,2025-06-07 00:00:00+00:00,NaN,0.603091,True
7,2025-06-08 00:00:00+00:00,0.602482,0.602482,False
8,2025-06-09 00:00:00+00:00,NaN,0.600019,True
9,2025-06-10 00:00:00+00:00,0.597555,0.597555,False


## Step (c): Weather forcing line separation

Operational (Open-Meteo, simulated) and archive (ERA5-Land, simulated) are kept in
**explicitly separate columns**, never silently merged, per
`docs/decisions/weather-forcing-split.md`. Column names are prefixed by line so a later join
cannot accidentally blend them.


In [4]:
operational_weather = pd.DataFrame({
    "date": dates,
    "operational__t_mean_c": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=1.0),
    "operational__solar_rad_mj_m2_day": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=1.5).clip(0, None),
    "operational__source": "open-meteo",
})

archive_weather = pd.DataFrame({
    "date": dates,
    "archive__t_mean_c": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=0.8),
    "archive__solar_rad_mj_m2_day": seasonal_signal(WINDOW_DAYS, base=24, amplitude=4, noise_std=1.2).clip(0, None),
    "archive__source": "era5-land",
})

# Explicit outer join on date only — column prefixes make it structurally impossible to
# confuse the two lines downstream (e.g. accidentally averaging operational and archive temps).
weather_lines = operational_weather.merge(archive_weather, on="date", how="outer")
assert not any(c.startswith("t_mean_c") for c in weather_lines.columns), \
    "weather forcing lines must never share an unprefixed column name"
weather_lines.head()


,date,operational__t_mean_c,operational__solar_rad_mj_m2_day,operational__source,archive__t_mean_c,archive__solar_rad_mj_m2_day,archive__source
0,2025-06-01 00:00:00+00:00,26.695476,27.025513,open-meteo,26.244839,25.915004,era5-land
1,2025-06-02 00:00:00+00:00,25.200842,25.416849,open-meteo,25.148049,28.383484,era5-land
2,2025-06-03 00:00:00+00:00,24.528133,23.150785,open-meteo,26.322940,27.122450,era5-land
3,2025-06-04 00:00:00+00:00,25.723837,24.911733,open-meteo,28.843945,27.013574,era5-land
4,2025-06-05 00:00:00+00:00,25.673577,26.279965,open-meteo,24.861704,26.108523,era5-land


## Assembled zone-day feature table

Column names follow `docs/schemas.md` conventions where a direct mapping exists
(`zone_id`, `theta_m3m3`-style units suffixes). This is illustrative only — the real ETL
asset lives in `packages/etl/`, not here.


In [5]:
zone_ids = points_df["zone_id"].unique()

rows = []
for zone_id in zone_ids:
    for i, date in enumerate(dates):
        rows.append({
            "zone_id": zone_id,
            "date_utc": date,
            "ndvi_filled": ndvi_filled.iloc[i],
            "operational__t_mean_c": operational_weather["operational__t_mean_c"].iloc[i],
            "operational__solar_rad_mj_m2_day": operational_weather["operational__solar_rad_mj_m2_day"].iloc[i],
            "archive__t_mean_c": archive_weather["archive__t_mean_c"].iloc[i],
        })

zone_day_df = pd.DataFrame(rows)
assert zone_day_df["zone_id"].nunique() in (2, 3, 4), "spatial-analysis-unit.md requires 2-4 zones per parcel"
zone_day_df.head()


,zone_id,date_utc,ndvi_filled,operational__t_mean_c,operational__solar_rad_mj_m2_day,archive__t_mean_c
0,zone-1,2025-06-01 00:00:00+00:00,0.639035,26.695476,27.025513,26.244839
1,zone-1,2025-06-02 00:00:00+00:00,0.639035,25.200842,25.416849,25.148049
2,zone-1,2025-06-03 00:00:00+00:00,0.598613,24.528133,23.150785,26.322940
3,zone-1,2025-06-04 00:00:00+00:00,0.602017,25.723837,24.911733,28.843945
4,zone-1,2025-06-05 00:00:00+00:00,0.605420,25.673577,26.279965,24.861704
